In [1]:
import pandas as pd
df = pd.read_csv('customer_shopping_behavior_info.csv')

In [2]:
df.head()
df.info()
df.describe(include='all')
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3900 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [3]:
#replace all mising values with median of each column
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))  
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [4]:
#makes headers all lowercase and replaces all spaces with "_"
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
df.columns 

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [5]:
#create a cloumn age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels) #splits into four and assigns them based of data
df[['age', 'age_group']].head(10)

,age,age_group
0,58,Senior
1,53,Middle-aged
2,39,Adult
3,41,Adult
4,40,Adult
5,67,Senior
6,65,Senior
7,68,Senior
8,66,Senior
9,58,Senior


In [6]:
# create cloumn purchase_frequency_days
frequency_mapping = {'Fortnightly': 14, 
                     'Weekly':7,
                     'Monthly': 30,
                     'Quarterly': 90,
                     'Bi-Weekly': 14,
                     'Annually': 365,
                     'Every 3 Months': 90
                     }
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping) 
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,90,Every 3 Months
1,7,Weekly
2,14,Bi-Weekly
3,90,Quarterly
4,90,Quarterly
5,365,Annually
6,365,Annually
7,14,Bi-Weekly
8,7,Weekly
9,90,Every 3 Months


In [7]:
#checks to see if both columns carry same info IF TRUE
df[['discount_applied', 'promo_code_used']].head(10) 
(df['discount_applied'] == df['promo_code_used']).all() 

np.True_

In [8]:
#Since TRUE lets remove promo_code_used
df = df.drop('promo_code_used', axis=1)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [9]:
!pip install pyodbc sqlalchemy 

In [10]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "sa"
password = "Ethiopia2127_V2"
host = "localhost"
port = "1433"
database = "customer_analysis"

driver = quote_plus("ODBC Driver 18 for SQL Server")

engine = create_engine(
    f"mssql+pyodbc://{username}:{password}@{host}:{port}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes"
)

df.to_sql("customer", engine, if_exists="replace", index=False)

pd.read_sql("SELECT TOP 5 * FROM customer;", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,58,Male,Scarf,Accessories,48,Florida,S,Lavender,Spring,4.0,No,Free Shipping,No,33,Venmo,Every 3 Months,Senior,90
1,2,53,Male,Jewelry,Accessories,73,Iowa,M,Purple,Spring,4.5,No,Next Day Air,No,10,Credit Card,Weekly,Middle-aged,7
2,3,39,Male,Jacket,Clothing,64,Rhode Island,M,Maroon,Summer,4.0,Yes,Next Day Air,No,19,Bank Transfer,Bi-Weekly,Adult,14
3,4,41,Male,Sunglasses,Accessories,49,Wyoming,M,Turquoise,Fall,4.5,No,Next Day Air,Yes,24,Credit Card,Quarterly,Adult,90
4,5,40,Male,Vest,Clothing,29,Rhode Island,L,Lavender,Fall,3.0,No,2-Day Shipping,Yes,15,Bank Transfer,Quarterly,Adult,90
